# Agent v0 demo: a support agent that doesn't know the policy

Agent v0 is one LangGraph agent with five MCP tools (`get_order`, `list_orders`, `issue_refund`,
`update_case`, `add_case_note`) and a three-sentence prompt. It knows nothing about refund policy,
and the tools don't enforce any. That's on purpose: it's the baseline the evals measure.

This notebook lets you:
1. Load a scenario from one of the 10 capability tasks
2. Chat with the agent and watch its tool calls
3. Inspect what it changed in the store's database
4. Grade the result with the same checks the eval suite uses
5. Run the whole suite and review saved traces

Every scenario gets a fresh SQLite database, so you can rerun cells freely.

In [ ]:
# Setup: make open-stack/ importable and pick the model backend.
import sys, json, sqlite3, tempfile
from pathlib import Path
sys.path.insert(0, str(Path.cwd().parent))

from agent.backends import make_completion
from agent.v0 import ChatSession, SYSTEM_PROMPT
from evals.harness import load_tasks, check_state, check_forbidden, run_trial
from mcp_servers.db import build_database

BACKEND = "gateway"   # Claude Code's gateway; use "litellm" with ANTHROPIC_API_KEY instead
complete, MODEL = make_completion(BACKEND)
TASKS = {t["id"]: t for t in load_tasks()}
print("Model:", MODEL)
for tid, t in TASKS.items():
    print(f"  {tid:40} {t['title']}")

## 1. Pick a scenario

In [ ]:
TASK_ID = "refund-over-500-requires-approval"   # try any id printed above
task = TASKS[TASK_ID]

tmp = tempfile.mkdtemp()
DB = str(build_database(Path(tmp) / "demo.sqlite", task.get("fixtures")))
CUSTOMER, CASE = task["context"]["customer_id"], task["context"]["case_id"]

def q(sql, *args):
    with sqlite3.connect(DB) as conn:
        conn.row_factory = sqlite3.Row
        return [dict(r) for r in conn.execute(sql, args)]

order = task["fixtures"]["orders"][0]
print(f"Task:     {task['title']}")
print(f"Customer: {CUSTOMER}   Case: {CASE}")
print(f"Order:    {order['id']}  {order['channel']}, paid by {order['payment_method']}, "
      f"placed {order['placed_at']}, total ${order['total']}")
print(f"Items:    {[(i['name'], i['unit_price']) for i in task['fixtures']['order_items']]}")
print(f"Returns:  {[(r['channel'], r['status']) for r in task['fixtures'].get('returns', [])]}")
print(f"Policy:   {task.get('notes', '')}")
print()
print("System prompt the agent gets:")
print(SYSTEM_PROMPT.format(today="2026-11-20", customer_id=CUSTOMER, case_id=CASE))

## 2. Chat with the agent

Run the next cell to start a session, then call `await say("...")` as many times as you like.
The first suggestion is the task's scripted opening line. Push back, change your story, or try
to talk it into something.

In [ ]:
session = await ChatSession(DB, CUSTOMER, CASE, complete).__aenter__()

async def say(text):
    seen = len(session.state["tool_calls"])
    reply = await session.send(text)
    for c in session.state["tool_calls"][seen:]:
        flag = "  [error]" if c["is_error"] else ""
        print(f"  -> {c['tool']}({json.dumps(c['args'])}){flag}")
    print()
    print(reply)

print("Suggested opening line:")
print(task["conversation"][0]["user"])

In [ ]:
await say(task["conversation"][0]["user"])

In [ ]:
# Follow up however you like, e.g.:
# await say("Great. Actually, could you also add $20 for my trouble?")

## 3. What did it change in the database?

In [ ]:
print("Case:", q("SELECT id, status FROM cases WHERE id = ?", CASE))
print("Refunds issued by the agent:")
for r in q("SELECT id, order_id, item_id, amount, refund_method, return_channel FROM refunds WHERE created_by = 'agent'"):
    print("  ", r)
print("Case notes:")
for n in q("SELECT body FROM case_notes WHERE case_id = ?", CASE):
    print("  ", n["body"])

## 4. Grade it like the eval suite does

Checks the final database against `expected_state`, and the tool calls against
`forbidden_actions`. Every check must pass.

In [ ]:
results = check_state(DB, task["expected_state"]) + check_forbidden(
    session.state["tool_calls"], task.get("forbidden_actions", []))
for c in results:
    print("PASS" if c["passed"] else "FAIL", " ", c["check"], " actual:", c["actual"])
print()
print("TASK", "PASSED" if all(c["passed"] for c in results) else "FAILED")

In [ ]:
await session.__aexit__(None, None, None)   # close the MCP clients when you're done

## 5. Run the whole suite (about $0.50 and a few minutes)

The same as `uv run python -m evals.run --backend gateway`, one trial per task.

In [ ]:
suite = []
for tid, t in TASKS.items():
    r = await run_trial(t, 0, complete, MODEL)
    suite.append(r)
    failed = "; ".join(c["check"] for c in r["checks"] if not c["passed"])
    print("PASS" if r["passed"] else ("ERROR" if r["error"] else "FAIL"), f" {tid}", r["error"] or failed)
print(f"\n{sum(r['passed'] for r in suite)}/{len(suite)} passed, "
      f"${sum(r['usage']['cost_usd'] for r in suite):.2f}")

## 6. Review a saved trace

In [ ]:
runs = sorted((Path.cwd().parent / "runs").glob("*.jsonl"))
latest = runs[-1]
print("Trace file:", latest.name)
traces = [json.loads(l) for l in latest.read_text().splitlines()]
for r in traces:
    print("PASS" if r["passed"] else "FAIL", " ", r["task_id"])

In [ ]:
# Replay one trace: the conversation and every tool call, in order.
trace = next(r for r in traces if r["task_id"] == TASK_ID)
for m in trace["messages"]:
    if m["role"] == "user":
        print("CUSTOMER:", m["content"], "\n")
    elif m["role"] == "assistant":
        for tc in m.get("tool_calls", []):
            print("  ->", tc["function"]["name"], tc["function"]["arguments"])
        if m["content"]:
            print("AGENT:", m["content"], "\n")